In [ ]:
# @title Licensed under the Apache License, Version 2.0
#
# Copyright 2026 Google LLC.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Legal Document Analysis with the Gemini API

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/legal-ai/Legal_Document_Analysis_with_Gemini.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/>
</a>

## Overview

This notebook demonstrates how to use the Gemini API to analyze legal documents. You will learn how to:

1. **Summarize** a legal document in plain language
2. **Extract key clauses** from a contract using structured output (JSON mode)
3. **Answer questions** about a legal document using document Q&A
4. **Compare** two documents side by side

These techniques are useful for legal tech applications, document review tools, and educational platforms like [SmoothOperationofLaw](https://smoothoperationoflaw.com).

> **Disclaimer**: This notebook is for educational and demonstration purposes only. AI-generated analysis of legal documents is not legal advice. Always consult a licensed attorney for advice about your specific legal situation.

## Setup

In [ ]:
%pip install -U -q 'google-genai>=1.0.0'

In [ ]:
import json
import os

from google import genai
from google.genai import types
from IPython.display import Markdown, display

To run this notebook, you need a Gemini API key. If you don't have one, get one free
at [Google AI Studio](https://aistudio.google.com/). Store it in the environment
variable `GEMINI_API_KEY`.

In [ ]:
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

In [ ]:
MODEL_ID = "gemini-2.5-flash"  # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-3.1-flash-lite-preview", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

## Sample document

For this notebook, use a sample consulting services agreement. In a real application, you would load documents from files or user uploads.

In [ ]:
sample_contract = """
CONSULTING SERVICES AGREEMENT

This Consulting Services Agreement ("Agreement") is entered into as of January 1, 2025,
between Acme Corp, a Delaware corporation ("Client"), and Jane Smith Consulting LLC,
a California limited liability company ("Consultant").

1. SERVICES
Consultant agrees to provide software architecture consulting services as described in
Exhibit A ("Services"). Consultant shall commence services on February 1, 2025,
and complete them by July 31, 2025.

2. COMPENSATION
Client shall pay Consultant $200 per hour. Consultant shall submit invoices monthly.
Payment is due within 30 days of invoice. Late payments accrue interest at 1.5% per month.

3. INTELLECTUAL PROPERTY
All work product created by Consultant under this Agreement shall be considered
work-made-for-hire and shall be the sole property of Client upon full payment.
Consultant retains rights to pre-existing intellectual property ("Background IP").

4. CONFIDENTIALITY
Consultant agrees to keep all Client information strictly confidential during and for
three (3) years after termination of this Agreement. This does not apply to information
that is publicly available or independently developed by Consultant.

5. TERMINATION
Either party may terminate this Agreement with 30 days written notice. Client may
terminate immediately for cause. Upon termination, Client shall pay for all services
rendered through the termination date.

6. LIMITATION OF LIABILITY
Consultant's total liability under this Agreement shall not exceed the fees paid in the
three (3) months preceding the claim. Neither party shall be liable for indirect,
incidental, or consequential damages.

7. GOVERNING LAW
This Agreement shall be governed by the laws of the State of California.
Any disputes shall be resolved by binding arbitration in San Francisco, CA.

8. INDEPENDENT CONTRACTOR
Consultant is an independent contractor, not an employee. Consultant is responsible
for all taxes and benefits.
"""

## 1. Summarize a document in plain language

Use Gemini to generate a plain-language summary that makes the contract accessible to non-lawyers.

In [ ]:
summary_prompt = """
You are a legal education assistant. Summarize the following contract in plain language
for someone without a legal background. Use simple sentences. Define any legal terms
you must use. Organize your summary with these headings:
- What is this contract?
- What does each party agree to do?
- Key dates and deadlines
- Money and payment terms
- How can the contract end?
- Important protections and limitations

Contract:
{contract}
""".format(contract=sample_contract)

response = client.models.generate_content(
    model=MODEL_ID,
    contents=summary_prompt,
    config=types.GenerateContentConfig(
        temperature=0.1,
    ),
)

display(Markdown(response.text))

## 2. Extract key clauses using structured output

Use JSON mode to extract specific clauses in a structured format — useful for populating a database or building a contract review dashboard.

In [ ]:
extraction_prompt = """
Extract key information from the following contract. Return a JSON object with these fields:
- parties: list of {name, role} objects
- effective_date: string (ISO date if present, else null)
- payment_terms: {rate, rate_unit, payment_due_days, late_fee_rate}
- termination: {notice_days, immediate_termination_for_cause: bool}
- confidentiality_period_years: number
- liability_cap: string (describe the cap)
- governing_law_jurisdiction: string
- dispute_resolution: string
- ip_ownership: string (brief description)

Contract:
{contract}
""".format(contract=sample_contract)

response = client.models.generate_content(
    model=MODEL_ID,
    contents=extraction_prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
    ),
)

clauses = json.loads(response.text)
print(json.dumps(clauses, indent=4))

## 3. Question answering about a document

Allow users to ask natural-language questions about the document.
This is the core of a legal document assistant.

In [ ]:
questions = [
    "Who owns the work that the consultant creates?",
    "What happens if the client pays late?",
    "Can the client fire the consultant immediately?",
]

for question in questions:
    qa_prompt = """
You are a legal education assistant. Answer the following question about the contract
below. Use plain language. If the contract doesn't address the question, say so clearly.
Do not speculate beyond what the document says. Do not provide legal advice.

Contract:
{document}

Question: {question}
""".format(document=sample_contract, question=question)

    qa_response = client.models.generate_content(
        model=MODEL_ID,
        contents=qa_prompt,
        config=types.GenerateContentConfig(
            temperature=0.1,
        ),
    )
    print(f"Q: {question}")
    print(f"A: {qa_response.text}")
    print("-" * 60)

## 4. Compare two documents

Compare two versions of a contract or two different agreements to highlight key differences — useful for redline review and negotiation.

In [ ]:
# A revised version with different terms for comparison
revised_contract = """
CONSULTING SERVICES AGREEMENT (REVISED)

This Agreement is entered into as of January 1, 2025, between Acme Corp ("Client")
and Jane Smith Consulting LLC ("Consultant").

1. SERVICES: Consultant shall provide services per Exhibit A from March 1, 2025
   to December 31, 2025.

2. COMPENSATION: Client shall pay $175 per hour. Invoices are due within 60 days.
   No interest on late payments.

3. INTELLECTUAL PROPERTY: Consultant retains ownership of all work product.
   Client receives a perpetual, non-exclusive license to use deliverables.

4. CONFIDENTIALITY: One (1) year confidentiality obligation post-termination.

5. TERMINATION: Either party may terminate with 14 days written notice.

6. LIMITATION OF LIABILITY: No cap on Consultant's liability.
   Consequential damages are permitted.

7. GOVERNING LAW: State of New York. Disputes resolved by litigation in New York courts.
"""

In [ ]:
comparison_prompt = """
Compare these two consulting agreements and explain the key differences in plain language.
Focus on terms that would matter most to the consultant. For each difference, indicate
which version is more favorable to the consultant and why.

Format your response as a table with columns:
Topic | Original Terms | Revised Terms | Better for Consultant?

Then provide a brief overall recommendation.

Agreement 1 (Original):
{original}

Agreement 2 (Revised):
{revised}
""".format(original=sample_contract, revised=revised_contract)

response = client.models.generate_content(
    model=MODEL_ID,
    contents=comparison_prompt,
    config=types.GenerateContentConfig(
        temperature=0.1,
    ),
)

display(Markdown(response.text))

## Next steps

You've seen four core legal document analysis patterns:

1. **Plain-language summarization** — makes documents accessible
2. **Structured clause extraction** — powers databases and dashboards
3. **Document Q&A** — the core of an interactive legal assistant
4. **Document comparison** — useful for contract negotiation and redline review

To go further:

- Use the **File API** to analyze large documents that exceed the prompt context window. See [`quickstarts/File_API.ipynb`](../../quickstarts/File_API.ipynb).
- Add **Grounding with Google Search** to verify cited statutes and case law. See [`quickstarts/Search_Grounding.ipynb`](../../quickstarts/Search_Grounding.ipynb).
- Build a **multi-turn conversation** so users can ask follow-up questions. See [`quickstarts/Chat.ipynb`](../../quickstarts/Chat.ipynb).
- Use **structured output with a schema** for stricter type validation. See [`quickstarts/JSON_mode.ipynb`](../../quickstarts/JSON_mode.ipynb).

---

**Disclaimer**: This notebook is for educational and demonstration purposes only. AI-generated analysis of legal documents is not legal advice. For advice about your specific legal situation, consult a licensed attorney.